# Lesson 8.2 - Responsible AI in Practice (checklists & toy tools)

## Objectives

- Build practical Responsible AI artifacts.
- Implement a basic group fairness helper.
- Capture decision logs for auditability.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Dict, List
import json
import pandas as pd


## Tool 1: Basic Group Fairness Statistics

This helper computes acceptance and error statistics by protected group for a toy binary decision task.


In [2]:
def group_fairness_report(df: pd.DataFrame, group_col: str, y_true: str, y_pred: str) -> pd.DataFrame:
    rows = []
    for g, part in df.groupby(group_col):
        approval_rate = part[y_pred].mean()
        accuracy = (part[y_true] == part[y_pred]).mean()
        false_positive_rate = ((part[y_pred] == 1) & (part[y_true] == 0)).mean()
        false_negative_rate = ((part[y_pred] == 0) & (part[y_true] == 1)).mean()
        rows.append(
            {
                "group": g,
                "n": len(part),
                "approval_rate": float(approval_rate),
                "accuracy": float(accuracy),
                "false_positive_rate": float(false_positive_rate),
                "false_negative_rate": float(false_negative_rate),
            }
        )
    out = pd.DataFrame(rows).sort_values("group").reset_index(drop=True)
    best_rate = out["approval_rate"].max()
    out["selection_rate_ratio_vs_best_group"] = out["approval_rate"] / best_rate if best_rate else 0.0
    return out


In [3]:
toy = pd.DataFrame(
    {
        "group": ["A","A","A","A","B","B","B","B"],
        "y_true": [1,0,1,0,1,1,0,0],
        "y_pred": [1,0,0,0,1,0,0,0],
    }
)

group_fairness_report(toy, "group", "y_true", "y_pred")


,group,n,approval_rate,accuracy,false_positive_rate,false_negative_rate,selection_rate_ratio_vs_best_group
0,A,4,0.25,0.75,0.0,0.25,1.0
1,B,4,0.25,0.75,0.0,0.25,1.0


## Tool 2: Risk Assessment Checklist Template


In [4]:
risk_checklist = {
    "use_case": "credit_scoring_model_v3",
    "risk_tier": "high",
    "affected_groups": ["new applicants", "thin-file applicants"],
    "critical_harms": [
        "disparate denial rates",
        "insufficient explanation for adverse action",
        "data provenance gaps",
    ],
    "mitigations": [
        "monthly subgroup fairness review",
        "adverse-action explanation QA",
        "human review for low-confidence cases",
    ],
    "owner": "ml-risk-committee",
    "review_cadence": "monthly",
}
print(json.dumps(risk_checklist, indent=2))


{
  "use_case": "credit_scoring_model_v3",
  "risk_tier": "high",
  "affected_groups": [
    "new applicants",
    "thin-file applicants"
  ],
  "critical_harms": [
    "disparate denial rates",
    "insufficient explanation for adverse action",
    "data provenance gaps"
  ],
  "mitigations": [
    "monthly subgroup fairness review",
    "adverse-action explanation QA",
    "human review for low-confidence cases"
  ],
  "owner": "ml-risk-committee",
  "review_cadence": "monthly"
}


## Tool 3: Decision Logging for Audit Trails


In [5]:
@dataclass
class DecisionLog:
    timestamp_utc: str
    request_id: str
    system_version: str
    risk_tier: str
    decision: str
    confidence: float
    human_review_required: bool
    notes: str


def make_log(request_id: str, decision: str, confidence: float, risk_tier: str) -> Dict[str, object]:
    return asdict(
        DecisionLog(
            timestamp_utc=datetime.utcnow().isoformat(),
            request_id=request_id,
            system_version="credit-v3.2.1",
            risk_tier=risk_tier,
            decision=decision,
            confidence=confidence,
            human_review_required=(risk_tier == "high" and confidence < 0.65),
            notes="auto-generated by demo logger",
        )
    )

log_example = make_log("REQ-901", "deny", 0.61, "high")
log_example


/tmp/ipykernel_576712/3646670394.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp_utc=datetime.utcnow().isoformat(),


{'timestamp_utc': '2026-07-03T05:45:20.362300',
 'request_id': 'REQ-901',
 'system_version': 'credit-v3.2.1',
 'risk_tier': 'high',
 'decision': 'deny',
 'confidence': 0.61,
 'human_review_required': True,
 'notes': 'auto-generated by demo logger'}

## Connect to Theory

- Data-level controls appear as provenance and consent artifacts.
- Model-level controls appear as fairness and reliability diagnostics.
- System-level controls appear as logging, review gates, and escalation logic.

Responsible AI becomes real when these checks are integrated into delivery workflows, not added as end-stage paperwork.


## Business Case Studies & Exceptions

### Case A: Bank Credit Scoring Launch

A bank introduced a launch gate requiring fairness report, explanation QA, and risk committee approval before deployment. Time-to-launch increased initially, but remediation incidents dropped.

### Case B: GenAI Content Assistant

A marketing GenAI product added policy filters and human approval for regulated claims. False negatives in safety filtering were handled through active incident labeling and retraining.

### Exceptions

- For low-impact internal copilots, controls can be lighter but should still include basic logging.
- High-impact uses require strict review cadence and clear accountable owners.


## Interview Questions & Answers

1. **Q: What is a practical Responsible AI workflow?**
   **A:** Risk tiering, pre-launch checks, controlled launch, monitoring, and remediation.
2. **Q: How do you test fairness quickly?**
   **A:** Compare subgroup outcomes and error rates, then investigate drivers.
3. **Q: What is the role of audit logs?**
   **A:** They provide traceability for decisions, incidents, and compliance evidence.
4. **Q: Why include human review gates?**
   **A:** To prevent automated harm in high-risk or uncertain cases.
5. **Q: What should a risk checklist include?**
   **A:** use case, harms, mitigations, owner, and review cadence.
6. **Q: How is Responsible AI tied to MLOps?**
   **A:** Through deployment gates, monitoring jobs, and incident workflows.
7. **Q: What is a common anti-pattern?**
   **A:** Declaring principles without measurable controls.
8. **Q: How do you handle low-confidence outputs?**
   **A:** Escalate to human review or safe fallback logic.
9. **Q: What metric alone is insufficient for fairness?**
   **A:** Overall accuracy; subgroup behavior must be inspected.
10. **Q: How do you communicate trade-offs to executives?**
   **A:** Compare risk-adjusted outcomes, not only speed or conversion.
